# Fixed-Token-Budget Experiment

**Question:** Our LoRA-trained model produces shorter captions (~28 words vs 61 for baseline). Is the CHAIR reduction real, or is the model just "winning" by saying less?

**Method:** Re-run all 4 conditions (Baseline, LoRA only, Grounding only, LoRA + Grounding) at **fixed token budgets** (64 and 128 tokens), so caption length is held constant across methods. If the methods still beat baseline at the same token budget, the hallucination reduction is genuinely independent of caption length.

**Three possible outcomes:**

1. **Methods still win at fixed length** → hallucination reduction is real, length is a side effect.
2. **Gap shrinks at fixed length** → some of the win comes from brevity; honest mixed framing.
3. **Gap disappears at fixed length** → CHAIR drop was mostly a length artifact; reframe as "calibrated brevity."

**Self-contained:** Run this notebook top-to-bottom without depending on any other. Setup cells (install / model / adapters / helpers) are copied from `stage4_comparison.ipynb`.

## 0. Install dependencies (uncomment + run once, then restart session)

In [ ]:
# # RUN ONCE then Runtime -> Restart session. Skip on subsequent runs.
# !pip install -q 'transformers>=4.47' 'accelerate>=0.33' 'tokenizers>=0.21'
# !pip install -q 'torchao>=0.16.0'
# !pip install -q peft bitsandbytes
# !pip install -q pandas pillow tqdm pycocotools spacy sentencepiece
# !python -m spacy download en_core_web_sm -q
# print('Done — Runtime → Restart session, then skip this cell.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 123.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Done — Runtime → Restart session, then skip this cell.


In [8]:
!pip install -q -U torchao peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.5 MB/s eta 0:00:00


## 1. Imports and Drive setup

In [2]:
import os, json, pickle, random, re, gc
from collections import defaultdict

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/llava_hallucination_heads'
COCO_DIR = f'{WORK_DIR}/coco'
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    raise RuntimeError(
        'No GPU.\nFix: Runtime → Change runtime type → A100 GPU\n'
        'Then: Runtime → Disconnect and delete runtime → reconnect → re-run.')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Mounted at /content/drive
Device: cuda
GPU:  NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 2. Load LLaVA-1.5-7B (fp16, eager attention)

In [9]:
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID  = 'llava-hf/llava-1.5-7b-hf'
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation='eager',   # required for output_attentions=True
    device_map={'': 0},
)
model.eval()
print(f'Base model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Base model loaded. VRAM: 28.26 GB


## 3. Resolve model constants

In [10]:
if hasattr(model, 'language_model'):
    lm = model.language_model
else:
    lm = model.model.language_model

text_cfg         = model.config.text_config
NUM_LAYERS       = text_cfg.num_hidden_layers
NUM_HEADS        = text_cfg.num_attention_heads
HEAD_DIM         = text_cfg.hidden_size // NUM_HEADS
IMAGE_TOKEN_ID   = model.config.image_token_index
vision_cfg       = model.config.vision_config
NUM_IMAGE_TOKENS = (vision_cfg.image_size // vision_cfg.patch_size) ** 2  # 576
PROMPT_TEMPLATE  = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'

print(f'Layers={NUM_LAYERS}, Heads/layer={NUM_HEADS}, head_dim={HEAD_DIM}')
print(f'IMAGE_TOKEN_ID={IMAGE_TOKEN_ID}, NUM_IMAGE_TOKENS={NUM_IMAGE_TOKENS}')

Layers=32, Heads/layer=32, head_dim=128
IMAGE_TOKEN_ID=32000, NUM_IMAGE_TOKENS=576


## 4. Load Stage 1 artifacts + 400 eval images

In [11]:
from pycocotools.coco import COCO

# Hallucination heads
with open(f'{WORK_DIR}/results/final_hallucination_heads.json') as f:
    final_list = json.load(f)

hal_heads_by_layer = defaultdict(set)
for h in final_list:
    hal_heads_by_layer[h['layer']].add(h['head'])

# Top-16 subset (by causal delta order — first 16 in final_list are most causal)
hal_heads_top16 = defaultdict(set)
for h in final_list[:16]:
    hal_heads_top16[h['layer']].add(h['head'])

# Per-image cache
with open(f'{WORK_DIR}/cache/screening_state.pkl', 'rb') as f:
    stage1_state = pickle.load(f)
per_image_cache = stage1_state['per_image_cache']

# Selected images + GT objects
with open(f'{WORK_DIR}/cache/selected_imgs.json') as f:
    img_meta = json.load(f)
selected_imgs     = img_meta['ids']
img_to_gt_objects = {int(k): set(v) for k, v in img_meta['gt_objects'].items()}

# Rebuild image paths
ann_path = f'{COCO_DIR}/annotations/instances_val2014.json'
coco     = COCO(ann_path)
img_dir  = f'{COCO_DIR}/val2014_subset'
img_meta_coco  = coco.loadImgs(selected_imgs)
img_id_to_path = {m['id']: f"{img_dir}/{m['file_name']}" for m in img_meta_coco}

cat_id_to_name  = {c['id']: c['name'].lower() for c in coco.loadCats(coco.getCatIds())}
ALL_COCO_OBJECTS = set(cat_id_to_name.values())

# Same held-out split as Stage 2/3 (last 40)
# === BUILD 400-IMAGE EVAL SET ===
# Random sample from COCO val2014, excluding the 200 Stage 1 images,
# requiring ≥2 GT object categories per image. Matches the headline n=400 setup.

import urllib.request

N_EVAL = 400
EVAL_SEED = 42
stage1_ids = set(selected_imgs)

# Find candidate images (val2014 minus Stage 1, with ≥2 object categories)
print('Finding candidate images...')
candidates = []
for img_id in coco.getImgIds():
    if img_id in stage1_ids:
        continue
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
    unique_cats = {a['category_id'] for a in anns if a.get('iscrowd', 0) == 0}
    if len(unique_cats) >= 2:
        candidates.append(img_id)
print(f'  Candidates: {len(candidates)}')

# Sample 400 deterministically
rng = random.Random(EVAL_SEED)
eval_ids = rng.sample(candidates, N_EVAL)

# Build path + GT mapping
img_info_map = {m['id']: m for m in coco.loadImgs(eval_ids)}
img_id_to_path = {}   # OVERWRITES the previous dict that pointed to Stage 1 images
img_to_gt_objs = {}
for img_id in eval_ids:
    m = img_info_map[img_id]
    img_id_to_path[img_id] = f'{img_dir}/{m["file_name"]}'
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
    img_to_gt_objs[img_id] = {
        cat_id_to_name[a['category_id']] for a in anns
        if a.get('iscrowd', 0) == 0 and a['category_id'] in cat_id_to_name
    }

# Download any missing images
missing = [i for i in eval_ids if not os.path.exists(img_id_to_path[i])]
print(f'Images to download: {len(missing)} / {N_EVAL}')
failed = []
for img_id in tqdm(missing, desc='Downloading'):
    fname = img_info_map[img_id]['file_name']
    url   = f'http://images.cocodataset.org/val2014/{fname}'
    dest  = img_id_to_path[img_id]
    try:
        urllib.request.urlretrieve(url, dest)
    except Exception as e:
        failed.append(img_id)
        print(f'  Failed {img_id}: {e}')

# Final eval lists — exclude any download failures
eval_images     = [i for i in eval_ids if os.path.exists(img_id_to_path[i])]
eval_gt_objects = [img_to_gt_objs[i] for i in eval_images]
print(f'\nReady: {len(eval_images)} / {N_EVAL} eval images (seed={EVAL_SEED})')
if failed:
    print(f'  Download failures: {len(failed)} (skipped)')

print(f'Hallucination heads : {len(final_list)}  (top-16 subset: {sum(len(v) for v in hal_heads_top16.values())})')
print(f'Eval images         : {len(eval_images)}')

loading annotations into memory...
Done (t=4.10s)
creating index...
index created!
Finding candidate images...
  Candidates: 31539
Images to download: 0 / 400


Downloading: 0it [00:00, ?it/s]


Ready: 400 / 400 eval images (seed=42)
Hallucination heads : 32  (top-16 subset: 16)
Eval images         : 400


## 5. Load Stage 2 LoRA adapter

In [12]:
from peft import PeftModel

ADAPTER_DIR = f'{WORK_DIR}/results/stage2_lora_adapter'
assert os.path.isdir(ADAPTER_DIR), (
    f'Stage 2 adapter not found at {ADAPTER_DIR}.\n'
    'Run Stage 2 first (stage2_lora.ipynb Cell 14).')

model_lora = PeftModel.from_pretrained(model, ADAPTER_DIR)
model_lora.eval()
print(f'LoRA adapter loaded from: {ADAPTER_DIR}')
print(f'VRAM after adapter: {torch.cuda.memory_allocated()/1e9:.2f} GB')

LoRA adapter loaded from: /content/drive/MyDrive/llava_hallucination_heads/results/stage2_lora_adapter
VRAM after adapter: 28.27 GB


## 6. Vocabulary and NLP setup

In [13]:
import spacy
nlp = spacy.load('en_core_web_sm')

COCO_SYNONYMS = {
    'person':        ['man','woman','people','boy','girl','child','guy','lady','kid',
                      'baby','player','rider','skier','surfer','snowboarder'],
    'car':           ['vehicle','automobile','sedan','suv'],
    'dog':           ['puppy','dogs'], 'cat': ['kitten','cats'],
    'tv':            ['television','monitor','screen'], 'couch': ['sofa'],
    'cell phone':    ['phone','cellphone','smartphone'],
    'dining table':  ['table','desk'], 'wine glass': ['glass'],
    'bicycle':       ['bike'], 'motorcycle': ['motorbike'],
    'airplane':      ['plane','jet'], 'potted plant': ['plant'],
    'laptop':        ['computer'], 'refrigerator': ['fridge'],
    'truck':         ['lorry'], 'boat': ['ship','sailboat'],
    'fire hydrant':  ['hydrant'], 'hot dog': ['hotdog'],
    'traffic light': ['stoplight'],
    'sports ball':   ['ball','football','soccer ball','basketball'],
    'baseball bat':  ['bat'], 'tennis racket': ['racket','racquet'],
}
MULTIWORD_ALIASES = {
    'hydrant':'fire hydrant','hotdog':'hot dog','stoplight':'traffic light',
    'bat':'baseball bat','racket':'tennis racket','racquet':'tennis racket',
}
OBJECT_VOCAB = set(ALL_COCO_OBJECTS)
for syns in COCO_SYNONYMS.values():
    OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD_ALIASES.keys())

print(f'Object vocab size: {len(OBJECT_VOCAB)}')

Object vocab size: 133


## 7. Visual token span + grounding score

In [14]:
def get_visual_token_span(input_ids):
    """
    Find image token positions in input_ids.
    Handles both transformers behaviours:
      - New (>=4.47): processor pre-expands to 576 tokens
      - Old (<4.47):  single placeholder, model expands internally
    Returns (img_start, img_end) into the attention KV dimension.
    """
    ids  = input_ids[0]
    mask = (ids == IMAGE_TOKEN_ID)
    n_ph = int(mask.sum().item())
    pos  = mask.nonzero(as_tuple=True)[0]

    if n_ph >= NUM_IMAGE_TOKENS:
        # New transformers: already expanded
        img_start = int(pos[0].item())
        img_end   = int(pos[-1].item()) + 1
    else:
        # Old transformers: single placeholder expands to NUM_IMAGE_TOKENS
        img_start = int(pos[0].item())
        img_end   = img_start + NUM_IMAGE_TOKENS

    return img_start, img_end


def compute_grounding_score(attentions, heads_by_layer, img_start, img_end):
    """
    Mean visual attention mass across the specified heads at the last query position.
    Simpler and more interpretable than Stage 3's entropy-weighted formula.
    attentions: tuple of [1, H, Q, K] tensors (one per layer)
    Returns scalar in [0, 1].
    """
    if img_end <= img_start or not attentions:
        return 0.0
    scores = []
    for layer_idx, heads in heads_by_layer.items():
        if layer_idx >= len(attentions) or attentions[layer_idx] is None:
            continue
        attn = attentions[layer_idx]   # [1, H, Q, K]
        for h in heads:
            if h >= attn.shape[1]:
                continue
            row       = attn[0, h, -1, :].float()          # [K]
            vis_mass  = row[img_start:img_end].sum().item()
            total     = row.sum().item()
            scores.append(vis_mass / max(total, 1e-9))
    return float(np.mean(scores)) if scores else 0.0


# Quick sanity check
_dummy = Image.open(img_id_to_path[eval_images[0]]).convert('RGB')
_inp   = processor(text=PROMPT_TEMPLATE, images=_dummy, return_tensors='pt')
_s, _e = get_visual_token_span(_inp['input_ids'])
del _dummy, _inp
print(f'Visual token span: [{_s}, {_e}) → {_e - _s} tokens (expect {NUM_IMAGE_TOKENS})')
assert _e - _s == NUM_IMAGE_TOKENS, 'Span width mismatch!'

Visual token span: [5, 581) → 576 tokens (expect 576)


## 8. Penalty token IDs (single-token object words)

In [15]:
def build_penalty_token_ids(tokenizer, vocab):
    """Token IDs for words in vocab that encode as a single token (with or without leading space)."""
    ids = set()
    for w in sorted(vocab):
        for form in [w, ' ' + w, w.capitalize(), ' ' + w.capitalize()]:
            toks = tokenizer.encode(form, add_special_tokens=False)
            if len(toks) == 1:
                ids.add(int(toks[0]))
    return sorted(ids)

PENALTY_IDS = build_penalty_token_ids(processor.tokenizer, OBJECT_VOCAB)
PENALTY_IDS_TENSOR = torch.tensor(PENALTY_IDS, dtype=torch.long, device=device)
print(f'Penalty token IDs: {len(PENALTY_IDS)} single-token object words')

Penalty token IDs: 94 single-token object words


## 9. Generation functions (gen_greedy + gen_with_penalty)

In [16]:
@torch.no_grad()
def gen_greedy(model_obj, image_path, max_new_tokens=80):
    """Standard greedy decode. Uses model.generate() for efficiency."""
    img    = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img,
                       return_tensors='pt').to(device, torch.float16)
    inputs['input_ids']      = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()
    out = model_obj.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, use_cache=True)
    gen_ids = out[0, inputs['input_ids'].shape[1]:]
    return processor.tokenizer.decode(gen_ids, skip_special_tokens=True)


@torch.no_grad()
def gen_with_penalty(model_obj, image_path, heads_map,
                     theta=0.08, alpha=8.0, max_new_tokens=80):
    """
    Greedy decode with visual grounding penalty.
    At each step, if the mean visual-attention mass of `heads_map` is below
    `theta`, object-word logits are reduced by alpha*(g - theta).
    """
    img    = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img,
                       return_tensors='pt').to(device, torch.float16)
    inputs['input_ids']      = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()

    # Compute visual span once from input_ids (no extra forward pass)
    img_start, img_end = get_visual_token_span(inputs['input_ids'])

    past_kv  = None
    cur_ids  = inputs['input_ids']
    cur_mask = inputs['attention_mask']
    generated = []
    eos_id = processor.tokenizer.eos_token_id

    for _ in range(max_new_tokens):
        if past_kv is None:
            # First step: pass full prompt + image
            out = model_obj(
                input_ids=cur_ids,
                attention_mask=cur_mask,
                pixel_values=inputs['pixel_values'],
                use_cache=True,
                past_key_values=None,
                output_attentions=True,
                return_dict=True,
            )
        else:
            # Subsequent steps: single new token, KV cached, no pixel_values
            out = model_obj(
                input_ids=cur_ids,
                attention_mask=cur_mask,
                use_cache=True,
                past_key_values=past_kv,
                output_attentions=True,
                return_dict=True,
            )

        logits  = out.logits[:, -1, :].float()  # [1, vocab]
        g       = compute_grounding_score(out.attentions, heads_map, img_start, img_end)

        if g < theta and len(PENALTY_IDS_TENSOR) > 0:
            penalty = alpha * (g - theta)        # negative value
            logits[:, PENALTY_IDS_TENSOR] += penalty

        next_id = int(logits.argmax(dim=-1).item())
        generated.append(next_id)

        if next_id == eos_id:
            break

        next_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        past_kv     = out.past_key_values
        cur_ids     = next_tensor
        cur_mask    = torch.cat(
            [cur_mask, torch.ones(1, 1, dtype=torch.long, device=device)], dim=1)

    return processor.tokenizer.decode(generated, skip_special_tokens=True)


print('Generation functions defined.')

Generation functions defined.


## 10. CHAIR scoring helpers

In [17]:
def find_content_words(caption, gt_objects):
    gt_norm = set(o.lower() for o in gt_objects)
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm:
            expanded_gt.update(syns)
    for alias, canonical in MULTIWORD_ALIASES.items():
        if canonical in gt_norm:
            expanded_gt.add(alias)

    doc = nlp(caption)
    obj_words, hall_words = [], []
    for tok in doc:
        w = tok.text.lower().strip()
        if tok.pos_ not in ('NOUN', 'PROPN') or len(w) < 2:
            continue
        canonical = MULTIWORD_ALIASES.get(w, w)
        if w in OBJECT_VOCAB or canonical in OBJECT_VOCAB:
            obj_words.append(w)
            if w not in expanded_gt and canonical not in expanded_gt:
                hall_words.append(w)
    return obj_words, hall_words


def score_chair(captions_gt):
    """captions_gt: list of (caption, gt_set). Returns CHAIRs, CHAIRi."""
    chairs_list, chairi_list = [], []
    for cap, gt in captions_gt:
        obj_w, hall_w = find_content_words(cap, gt)
        chairs_list.append(1 if hall_w else 0)
        chairi_list.append(len(hall_w) / max(len(obj_w), 1))
    return (float(np.mean(chairs_list)) if chairs_list else 0.0,
            float(np.mean(chairi_list)) if chairi_list else 0.0)


@torch.no_grad()
def pope_score(model_obj, images, gt_objects_list, n_per_image=6, seed=7):
    """
    Binary yes/no object-presence questions.
    3 positive (GT objects) + 3 negative (random non-GT) per image.
    NOTE: the grounding penalty targets object-word tokens, not 'yes'/'no',
    so Stage 3/4 penalty has minimal effect on POPE answers.
    """
    rng  = random.Random(seed)
    cats = list(ALL_COCO_OBJECTS)
    preds, labels = [], []
    model_obj.eval()

    for img_id, gt_set in tqdm(zip(images, gt_objects_list),
                               total=len(images), desc='POPE', leave=False):
        img_path = img_id_to_path.get(img_id)
        if not img_path or not os.path.exists(img_path):
            continue
        img = Image.open(img_path).convert('RGB')

        pos_obj = rng.sample(list(gt_set), min(n_per_image // 2, len(gt_set)))
        neg_obj = rng.sample([c for c in cats if c not in gt_set],
                             min(n_per_image // 2, len(cats)))

        for obj, lbl in [(o, 1) for o in pos_obj] + [(o, 0) for o in neg_obj]:
            q   = f'Is there a {obj} in the image? Please answer yes or no.'
            prm = f'USER: <image>\n{q}\nASSISTANT:'
            inp = processor(text=prm, images=img, return_tensors='pt').to(device, torch.float16)
            inp['input_ids']      = inp['input_ids'].long()
            inp['attention_mask'] = inp['attention_mask'].long()
            out = model_obj.generate(**inp, max_new_tokens=5, do_sample=False)
            ans = processor.tokenizer.decode(
                out[0, inp['input_ids'].shape[1]:], skip_special_tokens=True).lower()
            preds.append(1 if 'yes' in ans else 0)
            labels.append(lbl)

        torch.cuda.empty_cache()

    preds  = np.array(preds)
    labels = np.array(labels)
    acc    = (preds == labels).mean()
    tp = ((preds==1)&(labels==1)).sum()
    fp = ((preds==1)&(labels==0)).sum()
    fn = ((preds==0)&(labels==1)).sum()
    prec = tp / max(tp+fp, 1)
    rec  = tp / max(tp+fn, 1)
    f1   = 2*prec*rec / max(prec+rec, 1e-9)
    return {'accuracy': float(acc), 'f1': float(f1),
            'precision': float(prec), 'recall': float(rec),
            'yes_rate': float(preds.mean()), 'n': len(preds)}


print('Eval helpers defined.')

Eval helpers defined.


## 11. Constants for the grounding controller

In [18]:
# Same values used in the headline 4-way evaluation
THETA = 0.08    # grounding threshold
ALPHA = 4.0    # logit-penalty strength (reduced from 8.0 to prevent over-suppression)

CONDITIONS = [
    ('baseline', False, False),   # no LoRA, no grounding
    ('stage2',   True,  False),   # LoRA only
    ('stage3',   False, True),    # grounding only
    ('stage4',   True,  True),    # both
]
print(f'THETA={THETA}  ALPHA={ALPHA}')
print(f'Conditions: {[c[0] for c in CONDITIONS]}')

THETA=0.08  ALPHA=4.0
Conditions: ['baseline', 'stage2', 'stage3', 'stage4']


## 12. Fixed-token-budget experiment

Runs the 4 conditions at `max_new_tokens` ∈ {64, 128} on a deterministic random sample of 200 images.

**Key implementation detail:** The 200-image sample is built ONCE (seed=42), then re-used across both budgets and all 4 conditions. So differences across budgets / methods are purely from the intervention, not from sampling variance.

**Resumable:** Each (budget, condition) combination saves to JSON immediately after finishing. If Colab disconnects mid-run, just re-run this cell — it'll skip completed (budget, condition) pairs.

In [19]:
print(f'eval_images count: {len(eval_images)}')
print(f'eval_gt_objects count: {len(eval_gt_objects)}')
print(f'First 5 IDs: {eval_images[:5]}')
print(f'Last 5 IDs: {eval_images[-5:]}')

# Check how many actually have image files on disk
import os
missing = [i for i in eval_images if not os.path.exists(img_id_to_path[i])]
print(f'Images with file on disk: {len(eval_images) - len(missing)} / {len(eval_images)}')
if missing:
    print(f'  First 5 missing IDs: {missing[:5]}')

eval_images count: 400
eval_gt_objects count: 400
First 5 IDs: [293474, 465878, 419401, 432962, 183519]
Last 5 IDs: [497375, 568630, 321866, 371241, 1146]
Images with file on disk: 400 / 400


In [20]:
import os, json
WORK_DIR = '/content/drive/MyDrive/llava_hallucination_heads'

# Find the saved results file (auto-named by image count)
result_files = [f for f in os.listdir(f'{WORK_DIR}/results')
                if f.startswith('fixed_token_budget_n')]
print(f'Saved JSON files: {result_files}')

# Load it and check what's done
for fname in result_files:
    with open(f'{WORK_DIR}/results/{fname}') as f:
        r = json.load(f)
    print(f'\n{fname}:')
    for budget_key, conds in r.items():
        print(f'  {budget_key}: {sorted(conds.keys())}')

Saved JSON files: ['fixed_token_budget_n400.json']

fixed_token_budget_n400.json:
  budget_64: ['baseline', 'stage2', 'stage3', 'stage4']
  budget_128: ['baseline']


In [21]:
import time

ABLATION_N    = 400                             # was 200
SAMPLE_SEED   = 42                              # same seed
TOKEN_BUDGETS = [64, 128]
RESULTS_PATH  = f'{WORK_DIR}/results/fixed_token_budget_n400.json'   # was n200

rng = random.Random(SAMPLE_SEED)
sampled_idx  = sorted(rng.sample(range(len(eval_images)), ABLATION_N))
abl_imgs     = [eval_images[i]     for i in sampled_idx]
abl_gts      = [eval_gt_objects[i] for i in sampled_idx]

print(f'Using {len(abl_imgs)} images (seed={SAMPLE_SEED})')
print(f'First 5 img IDs: {abl_imgs[:5]}')
print(f'Last 5 img IDs:  {abl_imgs[-5:]}')

# Resume support
results = {}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f'Resuming. Already done: {sorted(results.keys())}')

# Ensure LoRA is at full strength
def set_lora_scale(model_obj, scale):
    for name, module in model_obj.named_modules():
        if hasattr(module, 'scaling') and isinstance(module.scaling, dict):
            for key in module.scaling:
                module.scaling[key] = scale
set_lora_scale(model_lora, 1.0)

for budget in TOKEN_BUDGETS:
    key = f'budget_{budget}'
    if key in results and all(c in results[key] for c, _, _ in CONDITIONS):
        print(f'\nSkipping budget={budget} (already done)')
        continue

    results.setdefault(key, {})
    print(f'\n========== Token budget = {budget} ==========')

    for cond_name, use_lora, use_grounding in CONDITIONS:
        if cond_name in results[key]:
            print(f'  Skipping {cond_name} (already done)')
            continue

        if use_lora:
            model_lora.enable_adapter_layers()
        else:
            model_lora.disable_adapter_layers()

        chairs_list, chairi_list = [], []
        actual_lens               = []
        t_start = time.time()

        for img_id, gt_set in tqdm(list(zip(abl_imgs, abl_gts)),
                                    desc=f'  budget={budget}  {cond_name}'):
            img_path = img_id_to_path[img_id]
            if use_grounding:
                cap = gen_with_penalty(model_lora, img_path, hal_heads_by_layer,
                                        theta=THETA, alpha=ALPHA,
                                        max_new_tokens=budget)
            else:
                cap = gen_greedy(model_lora, img_path, max_new_tokens=budget)

            obj_w, hall_w = find_content_words(cap, gt_set)
            chairs_list.append(1 if hall_w else 0)
            chairi_list.append(len(hall_w) / max(len(obj_w), 1))
            actual_lens.append(len(cap.split()))
            torch.cuda.empty_cache()

        elapsed = time.time() - t_start

        results[key][cond_name] = {
            'CHAIRs':     float(np.mean(chairs_list)),
            'CHAIRi':     float(np.mean(chairi_list)),
            'avg_length': float(np.mean(actual_lens)),
            'n':          len(chairs_list),
            'elapsed_s':  elapsed,
        }

        # SAVE AFTER EACH CONDITION
        with open(RESULTS_PATH, 'w') as f:
            json.dump(results, f, indent=2)

        r = results[key][cond_name]
        print(f'    CHAIRs={r["CHAIRs"]:.4f}  CHAIRi={r["CHAIRi"]:.4f}  '
              f'avg_len={r["avg_length"]:.1f}  ({elapsed/60:.1f}min)')

set_lora_scale(model_lora, 1.0)
model_lora.enable_adapter_layers()
gc.collect(); torch.cuda.empty_cache()
print(f'\nAll runs saved to: {RESULTS_PATH}')

Using 400 images (seed=42)
First 5 img IDs: [293474, 465878, 419401, 432962, 183519]
Last 5 img IDs:  [497375, 568630, 321866, 371241, 1146]
Resuming. Already done: ['budget_128', 'budget_64']

Skipping budget=64 (already done)

========== Token budget = 128 ==========
  Skipping baseline (already done)


  budget=128  stage2:   0%|          | 0/400 [00:00<?, ?it/s]

    CHAIRs=0.2225  CHAIRi=0.0947  avg_len=45.0  (27.9min)


  budget=128  stage3:   0%|          | 0/400 [00:00<?, ?it/s]

    CHAIRs=0.4925  CHAIRi=0.1734  avg_len=83.9  (35.1min)


  budget=128  stage4:   0%|          | 0/400 [00:00<?, ?it/s]

    CHAIRs=0.2150  CHAIRi=0.0917  avg_len=43.7  (24.1min)

All runs saved to: /content/drive/MyDrive/llava_hallucination_heads/results/fixed_token_budget_n400.json


## 13. Comparison table

Three columns: max_new_tokens = 64, max_new_tokens = 128, and (if available) the original max_new_tokens = 80 from the headline 4-way evaluation. Compares CHAIRs, CHAIRi, and the *actual* average caption length (which may be lower than the budget if EOS is hit early).

In [22]:
# Load original results for reference (if present)
orig_path = f'{WORK_DIR}/results/stage4_400img_results.json'
orig = {}
if os.path.exists(orig_path):
    with open(orig_path) as f:
        orig_data = json.load(f)
        orig = orig_data.get('chair', {})

# Re-load fixed-budget results (in case kernel was restarted)
with open(RESULTS_PATH) as f:
    results = json.load(f)

def get(budget, cond, field):
    return results.get(f'budget_{budget}', {}).get(cond, {}).get(field)

print('=' * 92)
print(f'{"":>14} {"":>12} {"max=64":>16} {"max=128":>16} {"max=80 (orig n=400)":>22}')
print('-' * 92)
for cond_name, _, _ in CONDITIONS:
    line = f'{cond_name:>14}  CHAIRs (↓): '
    for b in TOKEN_BUDGETS:
        v = get(b, cond_name, 'CHAIRs')
        line += f'  {v:>14.4f}' if v is not None else f'  {"—":>14}'
    v_orig = orig.get(cond_name, {}).get('CHAIRs')
    line += f'  {v_orig:>20.4f}' if v_orig is not None else f'  {"—":>20}'
    print(line)

print()
for cond_name, _, _ in CONDITIONS:
    line = f'{cond_name:>14}  CHAIRi (↓): '
    for b in TOKEN_BUDGETS:
        v = get(b, cond_name, 'CHAIRi')
        line += f'  {v:>14.4f}' if v is not None else f'  {"—":>14}'
    v_orig = orig.get(cond_name, {}).get('CHAIRi')
    line += f'  {v_orig:>20.4f}' if v_orig is not None else f'  {"—":>20}'
    print(line)

print()
for cond_name, _, _ in CONDITIONS:
    line = f'{cond_name:>14}  avg_len:    '
    for b in TOKEN_BUDGETS:
        v = get(b, cond_name, 'avg_length')
        line += f'  {v:>14.1f}' if v is not None else f'  {"—":>14}'
    line += f'  {"—":>20}'
    print(line)
print('=' * 92)

# Quick verdict
print('\nVerdict — does the CHAIR reduction survive at fixed length?\n')
for b in TOKEN_BUDGETS:
    base = get(b, 'baseline', 'CHAIRs')
    ours = get(b, 'stage4',   'CHAIRs')
    if base is None or ours is None:
        continue
    delta = (1 - ours/base) * 100
    print(f'  max_new_tokens={b}:  baseline {base:.3f} → ours {ours:.3f}  '
          f'(−{delta:.0f}% relative)')

                                      max=64          max=128    max=80 (orig n=400)
--------------------------------------------------------------------------------------------
      baseline  CHAIRs (↓):           0.2800          0.5125                0.3700
        stage2  CHAIRs (↓):           0.2225          0.2225                0.2650
        stage3  CHAIRs (↓):           0.2675          0.4925                0.3100
        stage4  CHAIRs (↓):           0.2150          0.2150                0.2300

      baseline  CHAIRi (↓):           0.1066          0.1794                0.1558
        stage2  CHAIRi (↓):           0.0940          0.0947                0.1043
        stage3  CHAIRi (↓):           0.1053          0.1734                0.1407
        stage4  CHAIRi (↓):           0.0909          0.0917                0.0958

      baseline  avg_len:                49.5            84.2                     —
        stage2  avg_len:                28.4            45.0             

## How to read the results

- **If max=64 and max=128 columns both show baseline > ours (lower CHAIRs for ours):**
  - Hallucination reduction is REAL and independent of caption length.
  - The shorter captions seen in the original run are a *side effect* of the method, not the cause of the CHAIR improvement.
  - Frame as: "Our method reduces hallucination AND tends toward more concise captions; the two are independent properties."

- **If the gap closes at max=64 but persists at max=128:**
  - Hallucination reduction is partly bounded by caption length.
  - Honest framing: "Our method's improvement is most pronounced when given more generation budget."

- **If the gap disappears at both fixed budgets:**
  - Most of the CHAIR drop came from brevity, not from invented objects.
  - Reframe as: "Our method induces calibrated brevity — the model knows when to stop, which incidentally reduces hallucination."

Either way, this experiment is publishable and necessary.